# 商品属性识别与生图提示词生成 — 完整链路演示\n\n## 链路概览\n```\n图片输入 → 商品属性识别(DeepSeek) → 结构化JSON → 生图提示词生成 → (可选)生图API\n```

## 1. 环境准备与配置加载

In [1]:
import os\nimport sys\nimport json\nfrom pathlib import Path\n\n# 确保项目根目录在 path 中\nsys.path.insert(0, os.getcwd())\n\n# ── 配置 DeepSeek API Key ──\n# 方式1: 环境变量\n# os.environ['DEEPSEEK_API_KEY'] = 'your-api-key'\n\n# 方式2: 直接赋值（演示用）\nDEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY', '')\n\nprint(f'DeepSeek API Key: {"已配置" if DEEPSEEK_API_KEY else "未配置（将使用 mock 模式）"}')

SyntaxError: unexpected character after line continuation character (1958877769.py, line 1)

## 2. 第一题：商品属性识别与卖点提炼

### 2.1 查看配置（品类映射表、禁用词、平台人设）

In [ ]:
from product_ai.config import (\n    CATEGORY_DIMENSION_MAP,\n    BANNED_WORDS,\n    PLATFORM_PERSONAS,\n    get_dimensions,\n)\n\nprint('=== 品类 → 卖点维度映射 ===')\nfor cat, dims in CATEGORY_DIMENSION_MAP.items():\n    print(f'  {cat}: {dims}')\n\nprint(f'\n=== 禁用空洞词 ({len(BANNED_WORDS)} 个) ===')\nprint(f'  {BANNED_WORDS[:10]}...')\n\nprint(f'\n=== 平台人设 ===')\nfor platform, persona in PLATFORM_PERSONAS.items():\n    print(f'  {platform}: {persona[:60]}...')\n\n# 测试维度匹配\nprint(f'\n=== 维度匹配测试 ===')\nprint(f'美妆 > 面部护肤 > 面膜: {get_dimensions("美妆 > 面部护肤 > 面膜")}')\nprint(f'3C数码 > 手机: {get_dimensions("3C数码 > 手机")}')\nprint(f'未知品类: {get_dimensions("其他")}')

### 2.2 查看数据模型（Schema）

In [ ]:
from product_ai.schema import ProductInfo, SellingPoint, PlatformCopy\n\n# 构造一个示例 ProductInfo 验证 Schema\nmock_selling_points = [\n    SellingPoint(dimension='成分', description='日本进口玻尿酸原液，纯度达98%', priority=1),\n    SellingPoint(dimension='功效', description='15分钟快速补水，锁水时长可达24小时', priority=2),\n    SellingPoint(dimension='质地', description='轻薄水润膜布，透气不闷痘', priority=3),\n]\n\nmock_platform_copy = PlatformCopy(\n    taobao=['限时特惠！玻尿酸补水面膜买2送1，手慢无！', '爆款返场！98%高纯度玻尿酸，补水看得见'],\n    jd=['日本进口玻尿酸原液，纯度98%远超行业标准，SGS认证', '15分钟极速补水，24H锁水科技，正品保障假一赔十'],\n    douyin=['谁懂啊这个面膜真的绝了！敷完皮肤像喝饱水一样', '姐妹们冲！这个玻尿酸面膜太上头了，素颜出门也自信']\n)\n\nmock_product = ProductInfo(\n    category='美妆 > 面部护肤 > 面膜',\n    product_name='XX玻尿酸补水面膜',\n    selling_points=mock_selling_points,\n    platform_copy=mock_platform_copy\n)\n\nprint(json.dumps(mock_product.model_dump(), ensure_ascii=False, indent=2))

### 2.3 查看 Prompt 模板

In [ ]:
from product_ai.prompt_templates import build_system_prompt\n\nsystem_prompt = build_system_prompt()\nprint(f'System Prompt 长度: {len(system_prompt)} 字符')\nprint(f'\n--- 前 800 字符 ---')\nprint(system_prompt[:800])

### 2.4 校验器测试

In [ ]:
from product_ai.validator import ProductValidator, validate_product\n\n# 测试1: 正常通过\nprint('=== 测试1: 正常卖点 ===')\ntry:\n    validate_product(mock_product)\n    print('✓ 校验通过')\nexcept Exception as e:\n    print(f'✗ 校验失败: {e}')\n\n# 测试2: 空洞卖点\nprint('\n=== 测试2: 空洞卖点（含禁用词） ===')\nbad_sp = [SellingPoint(dimension='品质', description='质量很好，性价比高', priority=1)]\nbad_product = ProductInfo(\n    category='美妆 > 护肤 > 面膜',\n    product_name='测试商品',\n    selling_points=bad_sp,\n    platform_copy=mock_platform_copy\n)\ntry:\n    validate_product(bad_product, strict=True)\n    print('✓ 校验通过')\nexcept Exception as e:\n    print(f'✗ 校验失败（预期行为）: {e}')\n\n# 测试3: non-strict 模式只记录不抛异常\nprint('\n=== 测试3: non-strict 模式 ===')\nvalidator = ProductValidator(strict=False)\nhit = validator.check_selling_points(bad_product)\nprint(f'命中的禁用词: {hit}')

### 2.5 真实 API 调用（需要 DeepSeek API Key）

In [2]:
from product_ai.pipeline import ProductPipeline, recognize_product\nfrom product_ai.config import APIConfig\n\n# 注释掉真实调用，演示时取消注释\nUSE_REAL_API = False  # 改为 True 启用真实 API 调用\n\nif USE_REAL_API and DEEPSEEK_API_KEY:\n    api_config = APIConfig(api_key=DEEPSEEK_API_KEY)\n    pipeline = ProductPipeline(api_config=api_config)\n    \n    # 替换为你的图片路径\n    result = pipeline.run(image_path='demo_product.jpg', mode='single')\n    \n    if result.success:\n        print('=== 识别成功 ===')\n        print(json.dumps(result.product.model_dump(), ensure_ascii=False, indent=2))\n        print(f'\n重试次数: {result.retries}')\n    else:\n        print(f'识别失败: {result.error}')\nelse:\n    print('跳过真实 API 调用（设置 USE_REAL_API=True 并配置 API Key 启用）')

SyntaxError: unexpected character after line continuation character (1076949314.py, line 1)

## 3. 第二题：参数化生图提示词生成

### 3.1 查看映射表配置

In [ ]:
from prompt_gen.config import (\n    MATERIAL_REFLECTION_MAP,\n    CATEGORY_SCENE_MAP,\n    CATEGORY_STYLE_TENDENCY,\n    USAGE_PRESETS,\n    infer_material,\n)\n\nprint('=== 材质 → 反射参数 ===')\nfor mat, refl in list(MATERIAL_REFLECTION_MAP.items())[:5]:\n    print(f'  {mat}: {refl[:50]}...')\nprint(f'  ... 共 {len(MATERIAL_REFLECTION_MAP)} 种材质')\n\nprint(f'\n=== 品类 → 场景风格 ===')\nfor cat, scene in CATEGORY_SCENE_MAP.items():\n    print(f'  {cat}: style={scene["style"]}, palette={scene["palette"]}')\n\nprint(f'\n=== 三种用途配置 ===')\nfor name, cfg in USAGE_PRESETS.items():\n    print(f'  {name}: ratio={cfg.ratio}, bg={cfg.background}, comp={cfg.composition}')\n\n# 材质推断测试\nprint(f'\n=== 材质推断测试 ===')\nsp = [{'dimension': '材质', 'description': '优质皮革面料'}]\nprint(f'推断材质: {infer_material("皮革手提包", "服饰", sp)}')

### 3.2 参数映射引擎演示

In [ ]:
from prompt_gen.mapper import ParameterMapper\n\nmapper = ParameterMapper()\n\n# 使用 mock 数据模拟 Task1 输出\ntask1_output = {\n    'product_name': 'XX玻尿酸补水面膜',\n    'category': '美妆 > 面部护肤 > 面膜',\n    'selling_points': [\n        {'dimension': '成分', 'description': '日本进口玻尿酸原液，纯度达98%', 'priority': 1},\n        {'dimension': '功效', 'description': '15分钟快速补水，锁水时长可达24小时', 'priority': 2},\n    ]\n}\n\n# 生成三种用途的参数\nfor usage in ['main', 'scene', 'selling_point']:\n    print(f'\n===== {usage} 用途参数 =====')\n    params = mapper.map(\n        product_name=task1_output['product_name'],\n        category=task1_output['category'],\n        selling_points=task1_output['selling_points'],\n        usage_type=usage,\n    )\n    for field, value in params:\n        if value:\n            print(f'  [{field}]: {value[:120]}...' if len(str(value)) > 120 else f'  [{field}]: {value}')

### 3.3 完整提示词组裝与对比

In [ ]:
from prompt_gen.pipeline import PromptGenPipeline\n\npipeline = PromptGenPipeline()\n\n# 一次性生成三种用途的完整提示词\nresults = pipeline.generate_all_usages(\n    product_name=task1_output['product_name'],\n    category=task1_output['category'],\n    selling_points=task1_output['selling_points'],\n)\n\nfor usage_type, prompt in results.items():\n    print(f'\n{"="*60}')\n    print(f'  {usage_type.upper()} — 生图提示词 ({len(prompt.full_prompt)} 字符)')\n    print(f'{"="*60}')\n    print(prompt.full_prompt[:400])\n    if len(prompt.full_prompt) > 400:\n        print(f'... (截断，全长 {len(prompt.full_prompt)} 字符)')

## 4. 端到端完整链路演示

In [ ]:
def end_to_end_demo() -> dict:\n    \"\"\"\n    完整链路:\n    1. 商品属性识别 (此处使用 mock 数据代替真实 API)\n    2. 生图提示词生成\n    3. 返回完整结果\n    \"\"\"\n    print(\"=\" * 50)\n    print(\"Step 1: 商品属性识别\")\n    print(\"=\" * 50)\n    \n    # Mock: 模拟 Task1 识别结果\n    product = {\n        \"category\": \"3C数码 > 手机 > 旗舰机\",\n        \"product_name\": \"XX Pro 5G 智能手机\",\n        \"selling_points\": [\n            {\"dimension\": \"芯片/核心配置\", \"description\": \"骁龙8 Gen4 旗舰芯片，台积电3nm工艺\", \"priority\": 1},\n            {\"dimension\": \"参数\", \"description\": \"200MP主摄+50MP超广角+50MP长焦，支持8K视频\", \"priority\": 2},\n            {\"dimension\": \"续航\", \"description\": \"5500mAh大电池，120W有线+50W无线快充\", \"priority\": 3},\n        ],\n        \"platform_copy\": {\n            \"taobao\": [\"限时特价！XX Pro 5G旗舰直降500，下单送耳机！\"],\n            \"jd\": [\"骁龙8Gen4旗舰芯，200MP徕卡影像，正品国行联保\"],\n            \"douyin\": [\"这手机拍照真的绝了！200MP像素太上头了\"]\n        }\n    }\n    print(json.dumps(product, ensure_ascii=False, indent=2))\n    \n    print(\"\\n\" + \"=\" * 50)\n    print(\"Step 2: 生图提示词生成\")\n    print(\"=\" * 50)\n    \n    from prompt_gen.pipeline import PromptGenPipeline\n    gen = PromptGenPipeline()\n    prompts = gen.generate_all_usages(\n        product_name=product['product_name'],\n        category=product['category'],\n        selling_points=product['selling_points'],\n    )\n    \n    for usage, p in prompts.items():\n        print(f\"\\n--- {usage} ({len(p.full_prompt)} chars) ---\")\n        print(p.full_prompt[:300] + \"...\\n\")\n    \n    print(\"=\" * 50)\n    print(\"链路完成!\")\n    print(\"=\" * 50)\n    \n    return {\"product\": product, \"prompts\": {k: v.full_prompt for k, v in prompts.items()}}\n\ndemo_result = end_to_end_demo()

## 5. 扩展性验证：新增品类只需加一行映射

In [ ]:
# 测试：新增「宠物用品」品类——只需在 prompt_gen/config.py 加映射规则\n# 这里验证框架本身不需要修改代码\n\nfrom prompt_gen.mapper import ParameterMapper\nfrom prompt_gen.assembler import PromptAssembler\n\nnew_category_test = {\n    'product_name': '天然猫砂',\n    'category': '宠物用品 > 猫砂',\n    'selling_points': [\n        {'dimension': '材质', 'description': '天然豆腐猫砂，可冲厕所'},\n    ]\n}\n\n# 即使宠物用品不在映射表中，也会使用默认值，不会报错\nmapper = ParameterMapper()\nassembler = PromptAssembler()\n\nparams = mapper.map(\n    product_name=new_category_test['product_name'],\n    category=new_category_test['category'],\n    selling_points=new_category_test['selling_points'],\n    usage_type='main',\n)\n\nprompt = assembler.assemble(params, usage_type='main', category=new_category_test['category'], product_name=new_category_test['product_name'])\nprint(f'✓ 新品类「宠物用品」生成的提示词 ({len(prompt.full_prompt)} 字符):')\nprint(prompt.full_prompt[:400] + '...')\nprint('\\n✓ 框架无需修改代码即可覆盖新品类（使用默认映射）')

## 总结\n\n1. **product_ai** 模块完成了从商品图片到结构化 JSON 的完整链路\n2. **prompt_gen** 模块实现了一套框架覆盖全品类的参数化生图提示词生成\n3. 新增品类只需在映射表中加一行，框架代码无需修改\n4. 两种模块可以独立使用，也可以串联成完整链路